In [1]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg21zn25_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 276 atoms, Mg126Zn150


In [2]:
slab = surface(alloy, (1, 0, 1), 2, vacuum=8)

sym = np.array(slab.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = slab.get_positions()[:, 2]
thick = z.max() - z.min()

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Ratio Zn/Mg: {zn/mg:.4f} (need {25/21:.4f})")
print(f"Stoichiometric: {'YES' if zn*21 == mg*25 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 552, Mg: 252, Zn: 300
Thickness: 16.2 A
Ratio Zn/Mg: 1.1905 (need 1.1905)
Stoichiometric: YES
Symmetric: False


In [3]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Using tolerance: {tol:.3f} A\n")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Using tolerance: 0.047 A

Total atomic layers: 104

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0        Zn3     0.000  |      103        Zn3     0.000    YES
      1        Zn3     0.148  |      102        Mg3     0.210 NO <---
      2        Mg3     0.358  |      101        Mg9     0.291 NO <---
      3        Mg9     0.439  |      100        Mg6     0.382 NO <---
      4        Mg6     0.530  |       99        Zn6     0.445 NO <---
      5        Zn6     0.593  |       98        Zn6     0.821    YES
      6        Zn6     0.969  |       97        Zn3     0.920 NO <---
      7        Zn3     1.068  |       96        Mg3     1.019 NO <---
      8        Mg3     1.167  |       95        Mg6     1.336 NO <---
      9        Mg6     1.484  |       94       Zn12     1.421 NO <---
     10       Zn12     1.569  |       93        Mg3     1.646 NO <---
     11        Mg3     1.794  

In [4]:
print("Searching for symmetric removals...\n")

found_any = False
for bot_rm in range(0, 12):
    for top_rm in range(0, 12):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        tr = slab[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,0,1), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                mg_tr = sum(sym_tr == 'Mg')
                zn_tr = sum(sym_tr == 'Zn')
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{mg_tr} Zn{zn_tr}, removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
                found_any = True
        except:
            continue

if not found_any:
    print("No symmetric sub-slab found with up to 11 layers removed from each end")

Searching for symmetric removals...

  bot_rm=1, top_rm=0: 549 atoms, Mg252 Zn297, removed 3+0=3
  bot_rm=2, top_rm=1: 543 atoms, Mg252 Zn291, removed 6+3=9
  bot_rm=3, top_rm=2: 537 atoms, Mg246 Zn291, removed 9+6=15
  bot_rm=4, top_rm=3: 519 atoms, Mg228 Zn291, removed 18+15=33
  bot_rm=5, top_rm=4: 507 atoms, Mg216 Zn291, removed 24+21=45
  bot_rm=6, top_rm=5: 495 atoms, Mg216 Zn279, removed 30+27=57
  bot_rm=7, top_rm=6: 483 atoms, Mg216 Zn267, removed 36+33=69
  bot_rm=8, top_rm=7: 477 atoms, Mg216 Zn261, removed 39+36=75
  bot_rm=9, top_rm=8: 471 atoms, Mg210 Zn261, removed 42+39=81
  bot_rm=10, top_rm=9: 459 atoms, Mg198 Zn261, removed 48+45=93
  bot_rm=11, top_rm=10: 435 atoms, Mg198 Zn237, removed 60+57=117


In [5]:
# Remove bottom 1 layer only (3 Zn)
removed_bot = layer_idx[0]
keep = []
for j in range(1, n):
    keep.extend(layer_idx[j])

trimmed = slab[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

print(f"\nRemoved bottom ({len(removed_bot)} atoms):")
for j in removed_bot:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Check partners
ps_orig = AseAtomsAdaptor().get_structure(slab)
keep_set = set(keep)

for j in removed_bot:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    # Check if partner exists in kept slab
    dists = np.linalg.norm(slab.get_positions() - inv_cart, axis=1)
    min_d = min(dists[k] for k in keep_set)
    closest = min(keep_set, key=lambda k: dists[k])
    
    if min_d < 0.5:
        print(f"  idx={j} -> paired with idx={closest} dist={min_d:.3f}")
    else:
        print(f"  idx={j} -> UNPAIRED, inv at ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

SG = P-1
Inversion: f -> [0.07108434 0.46446078 0.0045819 ] - f

Removed bottom (3 atoms):
  idx=240 Zn (13.313, 7.667, 8.000)
  idx=49 Zn (26.626, 15.334, 8.000)
  idx=2 Zn (27.560, 0.000, 8.000)
  idx=240 -> UNPAIRED, inv at (21.955, 3.016, 24.383)
  idx=49 -> UNPAIRED, inv at (21.021, 18.351, 24.383)
  idx=2 -> UNPAIRED, inv at (7.708, 10.683, 24.383)


In [6]:
# 3 unpaired is odd — need supercell to make even.
# 1x2 gives 6, splits 3+3.

slab_2y = make_supercell(slab, [[1,0,0],[0,2,0],[0,0,1]])

sym2 = np.array(slab_2y.get_chemical_symbols())
mg2, zn2 = sum(sym2 == 'Mg'), sum(sym2 == 'Zn')
z2 = slab_2y.get_positions()[:, 2]
order2 = np.argsort(z2)

print(f"1x2 supercell: {len(slab_2y)} atoms, Mg{mg2} Zn{zn2}")
print(f"Stoichiometric: {'YES' if zn2*21 == mg2*25 else 'NO'}")

# Cluster layers
z_sorted2 = np.sort(z2)
gaps2 = np.diff(z_sorted2)
tol2 = max(0.02, min(0.3, np.median(gaps2[gaps2 > 0.01]) / 2))

layers2 = []
layer_idx2 = []
cur = [order2[0]]
for i in range(1, len(order2)):
    if z2[order2[i]] - z2[order2[i-1]] < tol2:
        cur.append(order2[i])
    else:
        layers2.append(Counter(sym2[j] for j in cur))
        layer_idx2.append(list(cur))
        cur = [order2[i]]
layers2.append(Counter(sym2[j] for j in cur))
layer_idx2.append(list(cur))
n2 = len(layers2)

# Remove bottom 1 layer
removed_bot2 = layer_idx2[0]
keep2 = []
for j in range(1, n2):
    keep2.extend(layer_idx2[j])

trimmed2 = slab_2y[keep2]
ps_trim2 = AseAtomsAdaptor().get_structure(trimmed2)

slab_trim2 = Slab(ps_trim2.lattice, ps_trim2.species, ps_trim2.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_trim2, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"After removing bottom layer: {len(trimmed2)} atoms, Symmetric = {slab_trim2.is_symmetric()}")

if slab_trim2.is_symmetric():
    sga2 = SpacegroupAnalyzer(ps_trim2, symprec=0.1)
    print(f"SG = {sga2.get_space_group_symbol()}")
    for op in sga2.get_symmetry_operations():
        if op.rotation_matrix[2][2] < 0:
            inv_translation = op.translation_vector
            print(f"Inversion: f -> {inv_translation} - f")
            break
    
    print(f"\nRemoved: {len(removed_bot2)} atoms")
    
    # Find unpaired
    ps_orig2 = AseAtomsAdaptor().get_structure(slab_2y)
    keep_set2 = set(keep2)
    unpaired = []
    for j in removed_bot2:
        frac = ps_orig2[j].frac_coords
        inv_frac = (inv_translation - frac) % 1.0
        inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)
        dists = np.linalg.norm(slab_2y.get_positions() - inv_cart, axis=1)
        min_d = min(dists[k] for k in keep_set2)
        if min_d > 0.5:
            unpaired.append(j)
    
    print(f"Unpaired: {len(unpaired)}")
    for j in unpaired:
        pos = slab_2y.positions[j]
        frac = ps_orig2[j].frac_coords
        inv_frac = (inv_translation - frac) % 1.0
        inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)
        print(f"  idx={j} {sym2[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) "
              f"-> inv: ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

1x2 supercell: 1104 atoms, Mg504 Zn600
Stoichiometric: YES
After removing bottom layer: 1098 atoms, Symmetric = True
SG = P-1
Inversion: f -> [0.07108434 0.73222749 0.0045819 ] - f

Removed: 6 atoms
Unpaired: 6
  idx=240 Zn (13.313, 7.667, 8.000) -> inv: (34.334, 26.018, 24.383)
  idx=792 Zn (25.691, 30.669, 8.000) -> inv: (21.955, 3.016, 24.383)
  idx=49 Zn (26.626, 15.334, 8.000) -> inv: (21.021, 18.350, 24.383)
  idx=554 Zn (12.379, 23.002, 8.000) -> inv: (7.708, 10.683, 24.383)
  idx=2 Zn (-0.000, 0.000, 8.000) -> inv: (20.087, 33.685, 24.383)
  idx=601 Zn (39.004, 38.336, 8.000) -> inv: (33.400, 41.352, 24.383)


In [7]:
# Find mutual pairs among the 6 unpaired
# Each bottom atom's inv should land near another bottom atom's position

mutual = []
used = set()
for i, j1 in enumerate(unpaired):
    if j1 in used:
        continue
    frac1 = ps_orig2[j1].frac_coords
    inv1 = ps_orig2.lattice.get_cartesian_coords((inv_translation - frac1) % 1.0)
    for j2 in unpaired[i+1:]:
        if j2 in used:
            continue
        # Check if inv(j1) lands near j2's position
        d1 = np.linalg.norm(slab_2y.positions[j2] - inv1)
        # Also check reverse
        frac2 = ps_orig2[j2].frac_coords
        inv2 = ps_orig2.lattice.get_cartesian_coords((inv_translation - frac2) % 1.0)
        d2 = np.linalg.norm(slab_2y.positions[j1] - inv2)
        
        if d1 < 0.5 or d2 < 0.5:
            mutual.append((j1, j2))
            used.add(j1)
            used.add(j2)
            print(f"  Mutual pair: ({j1}, {j2}) dist={min(d1,d2):.3f}")
            break

print(f"\nMutual pairs: {len(mutual)}")

if len(mutual) == 3:
    # Reconstruct: keep first, move second to inv(first)
    sc_fixed = slab_2y.copy()
    for keep_idx, move_idx in mutual:
        frac = ps_orig2[keep_idx].frac_coords
        inv_frac = (inv_translation - frac) % 1.0
        inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)
        old = sc_fixed.positions[move_idx].copy()
        sc_fixed.positions[move_idx] = inv_cart
        print(f"  Moved {move_idx} -> inv({keep_idx}): "
              f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
              f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

    sym_f = np.array(sc_fixed.get_chemical_symbols())
    mg_f = sum(sym_f == 'Mg')
    zn_f = sum(sym_f == 'Zn')

    ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
    slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
        miller_index=(1,0,1), oriented_unit_cell=ps_fixed, shift=0,
        scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

    print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
    print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
    print(f"Symmetric: {slab_fixed.is_symmetric()}")
else:
    print("Mutual pairing failed — trying brute force...")
    
    for keep3 in combinations(unpaired, 3):
        move3 = [j for j in unpaired if j not in keep3]
        for perm in permutations(keep3):
            sc_test = slab_2y.copy()
            for m, k in zip(move3, perm):
                frac = ps_orig2[k].frac_coords
                inv_frac = (inv_translation - frac) % 1.0
                inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)
                sc_test.positions[m] = inv_cart
            
            try:
                ps_test = AseAtomsAdaptor().get_structure(sc_test)
                slab_test = Slab(ps_test.lattice, ps_test.species, ps_test.frac_coords,
                    miller_index=(1,0,1), oriented_unit_cell=ps_test, shift=0,
                    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
                if slab_test.is_symmetric():
                    print(f"\nFOUND: keep={list(keep3)}, move={move3}")
                    sc_fixed = sc_test
                    sym_f = np.array(sc_fixed.get_chemical_symbols())
                    mg_f = sum(sym_f == 'Mg')
                    zn_f = sum(sym_f == 'Zn')
                    print(f"Atoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
                    print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
                    print(f"Symmetric: True")
                    break
            except:
                continue
        else:
            continue
        break
    else:
        print("No solution found")


Mutual pairs: 0
Mutual pairing failed — trying brute force...

FOUND: keep=[np.int64(240), np.int64(792), np.int64(49)], move=[np.int64(554), np.int64(2), np.int64(601)]
Atoms: 1104, Mg: 504, Zn: 600
Stoichiometric: YES
Symmetric: True


In [8]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [9]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg21zn25_101_2layers_reconstructed_ase.data")

print(f"Saved: slabs/mg21zn25_101_2layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg21zn25_101_2layers_reconstructed_ase.data
  Atoms: 1104
  Thickness: 16.4 A
  Area: 1267.8 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [10]:
#unrelaxed surface energy calculation
E_slab =      -1347.8961    # eV
E_bulk_per_fu = -363.4709  / 6  # eV per formula unit = 
n = 1104 / 46                # formula units in slab =
A = 1267.8              # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -60.578483 eV
n * E_bulk = -1453.88360 eV
E_slab - n*E_bulk = 105.98750 eV
S = 0.041800 eV/Ang^2
S = 0.6697 J/m^2


In [15]:
#relaxed surface energy calculation
E_slab_relaxed =  -1356.56596728944  # eV
E_bulk_per_fu =  -363.4709  / 6  # eV per formula unit
n = 1104 / 46                      # 32 formula units
A = 1267.8                 # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.6697
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 97.31763 eV
S relaxed = 0.038381 eV/Ang^2
S relaxed = 0.6149 J/m^2

Unrelaxed S = 0.6697 J/m^2
Relaxed S   = 0.6149 J/m^2
Relaxation energy = 0.0548 J/m^2


In [12]:
slab4 = surface(alloy, (1, 0, 1), 4, vacuum=8)
slab4_2y = make_supercell(slab4, [[1,0,0],[0,2,0],[0,0,1]])

z = slab4_2y.get_positions()[:, 2]
sym = np.array(slab4_2y.get_chemical_symbols())
order = np.argsort(z)
print(f"Atoms: {len(slab4_2y)}, Thickness: {z.max()-z.min():.1f} A")

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))
n = len(layers)
print(f"Atomic layers: {n}")

# Remove bottom 1 layer
removed_bot = layer_idx[0]
keep = []
for j in range(1, n):
    keep.extend(layer_idx[j])

trimmed = slab4_2y[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

try:
    slab_trim = Slab(ps_trim.lattice, ps_trim.species, ps_trim.frac_coords,
        miller_index=(1,0,1), oriented_unit_cell=ps_trim, shift=0,
        scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
    print(f"After removing bottom layer: Symmetric = {slab_trim.is_symmetric()}")
except:
    print("Symmetry check failed, trying other removals...")

if not slab_trim.is_symmetric():
    # Search for correct removal
    print("Searching...")
    for bot_rm in range(0, 8):
        for top_rm in range(0, 8):
            if bot_rm == 0 and top_rm == 0:
                continue
            k = []
            for j in range(bot_rm, n - top_rm):
                k.extend(layer_idx[j])
            if len(k) == 0:
                continue
            tr = slab4_2y[k]
            try:
                ps_tr = AseAtomsAdaptor().get_structure(tr)
                slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                    miller_index=(1,0,1), oriented_unit_cell=ps_tr, shift=0,
                    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
                if slab_tr.is_symmetric():
                    removed_bot_n = sum(len(layer_idx[j]) for j in range(bot_rm))
                    removed_top_n = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                    print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                          f"removed {removed_bot_n}+{removed_top_n}")
                    trimmed = tr
                    ps_trim = ps_tr
                    keep = k
                    removed_bot = []
                    for jj in range(bot_rm):
                        removed_bot.extend(layer_idx[jj])
                    break
            except:
                continue
        else:
            continue
        break

# Get inversion
sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")
for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

# Find removed atoms and unpaired
removed_all = [i for i in range(len(slab4_2y)) if i not in set(keep)]
print(f"\nRemoved: {len(removed_all)} atoms")

ps_orig = AseAtomsAdaptor().get_structure(slab4_2y)
keep_set = set(keep)
unpaired = []
for j in removed_all:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    dists = np.linalg.norm(slab4_2y.get_positions() - inv_cart, axis=1)
    min_d = min(dists[k] for k in keep_set)
    if min_d > 0.5:
        unpaired.append(j)

print(f"Unpaired: {len(unpaired)}")

# Auto-pair unpaired by mutual inversion
used = set()
mutual = []
for i, j1 in enumerate(unpaired):
    if j1 in used:
        continue
    frac1 = ps_orig[j1].frac_coords
    inv1 = ps_orig.lattice.get_cartesian_coords((inv_translation - frac1) % 1.0)
    for j2 in unpaired[i+1:]:
        if j2 in used:
            continue
        if np.linalg.norm(slab4_2y.positions[j2] - inv1) < 0.5:
            mutual.append((j1, j2))
            used.add(j1)
            used.add(j2)
            break

print(f"Mutual pairs: {len(mutual)}")

if len(mutual) == len(unpaired) // 2:
    # Reconstruct
    sc_fixed4 = slab4_2y.copy()
    for keep_idx, move_idx in mutual:
        frac = ps_orig[keep_idx].frac_coords
        inv_frac = (inv_translation - frac) % 1.0
        inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
        sc_fixed4.positions[move_idx] = inv_cart
        print(f"  Moved {move_idx} -> inv({keep_idx})")
    
    sym_f = np.array(sc_fixed4.get_chemical_symbols())
    mg_f = sum(sym_f == 'Mg')
    zn_f = sum(sym_f == 'Zn')
    
    ps_fixed4 = AseAtomsAdaptor().get_structure(sc_fixed4)
    slab_fixed4 = Slab(ps_fixed4.lattice, ps_fixed4.species, ps_fixed4.frac_coords,
        miller_index=(1,0,1), oriented_unit_cell=ps_fixed4, shift=0,
        scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
    
    print(f"\nAtoms: {len(sc_fixed4)}, Mg: {mg_f}, Zn: {zn_f}")
    print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
    print(f"Symmetric: {slab_fixed4.is_symmetric()}")
else:
    # Brute force
    print("Mutual pairing failed — trying brute force...")
    
    mg_unp = [j for j in unpaired if sym[j] == 'Mg']
    zn_unp = [j for j in unpaired if sym[j] == 'Zn']
    n_mg = len(mg_unp)
    n_zn = len(zn_unp)
    half_mg = n_mg // 2
    half_zn = n_zn // 2
    
    found = False
    count = 0
    for keep_mg in combinations(mg_unp, half_mg) if half_mg > 0 else [()]:
        move_mg = [i for i in mg_unp if i not in keep_mg]
        for keep_zn in combinations(zn_unp, half_zn) if half_zn > 0 else [()]:
            move_zn = [i for i in zn_unp if i not in keep_zn]
            for perm_mg in permutations(keep_mg) if half_mg > 0 else [()]:
                for perm_zn in permutations(keep_zn) if half_zn > 0 else [()]:
                    sc_test = slab4_2y.copy()
                    for m, k in zip(move_mg, perm_mg):
                        frac = ps_orig[k].frac_coords
                        inv_frac = (inv_translation - frac) % 1.0
                        inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
                        sc_test.positions[m] = inv_cart
                    for m, k in zip(move_zn, perm_zn):
                        frac = ps_orig[k].frac_coords
                        inv_frac = (inv_translation - frac) % 1.0
                        inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
                        sc_test.positions[m] = inv_cart
                    
                    count += 1
                    if count % 5000 == 0:
                        print(f"  tested {count}...")
                    try:
                        ps_test = AseAtomsAdaptor().get_structure(sc_test)
                        slab_test = Slab(ps_test.lattice, ps_test.species, ps_test.frac_coords,
                            miller_index=(1,0,1), oriented_unit_cell=ps_test, shift=0,
                            scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
                        if slab_test.is_symmetric():
                            print(f"\nFOUND after {count} tries!")
                            sc_fixed4 = sc_test
                            sym_f = np.array(sc_fixed4.get_chemical_symbols())
                            mg_f = sum(sym_f == 'Mg')
                            zn_f = sum(sym_f == 'Zn')
                            print(f"Atoms: {len(sc_fixed4)}, Mg: {mg_f}, Zn: {zn_f}")
                            print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
                            found = True
                            break
                    except:
                        continue
                if found:
                    break
            if found:
                break
        if found:
            break
    
    if not found:
        print(f"No solution found ({count} tested)")

# Print final thickness and area
if 'sc_fixed4' in dir():
    thick4 = sc_fixed4.get_positions()[:,2].max() - sc_fixed4.get_positions()[:,2].min()
    area4 = np.linalg.norm(np.cross(sc_fixed4.cell[0], sc_fixed4.cell[1]))
    print(f"Thickness: {thick4:.1f} A")
    print(f"Area: {area4:.1f} A²")

Atoms: 2208, Thickness: 32.6 A
Atomic layers: 208
After removing bottom layer: Symmetric = True
SG = P-1
Inversion: f -> [0.80883864 0.79779006 0.00303951] - f

Removed: 6 atoms
Unpaired: 6
Mutual pairs: 0
Mutual pairing failed — trying brute force...

FOUND after 1 tries!
Atoms: 2208, Mg: 1008, Zn: 1200
Stoichiometric: YES
Thickness: 32.8 A
Area: 1267.8 A²


In [13]:
ps = AseAtomsAdaptor().get_structure(sc_fixed4)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg21zn25_101_4layers_reconstructed_ase.data")

print(f"Saved: slabs/mg21zn25_101_4layers_reconstructed_ase.data")
print(f"  Atoms: 2208")
print(f"  Thickness: 32.8 A")
print(f"  Area: 1267.8 A²")

Saved: slabs/mg21zn25_101_4layers_reconstructed_ase.data
  Atoms: 2208
  Thickness: 32.8 A
  Area: 1267.8 A²


In [14]:
#unrelaxed surface energy calculation
E_slab =      -2796.5045   # eV
E_bulk_per_fu = -363.4709  / 6  # eV per formula unit = 
n = 2208 / 46                # formula units in slab =
A = 1267.8              # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -60.578483 eV
n * E_bulk = -2907.76720 eV
E_slab - n*E_bulk = 111.26270 eV
S = 0.043880 eV/Ang^2
S = 0.7030 J/m^2


In [16]:
#relaxed surface energy calculation
E_slab_relaxed = -2809.78009010352   # eV
E_bulk_per_fu =  -363.4709  / 6  # eV per formula unit
n = 2208 / 46                      # 32 formula units
A = 1267.8                 # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.7030
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 97.98711 eV
S relaxed = 0.038645 eV/Ang^2
S relaxed = 0.6192 J/m^2

Unrelaxed S = 0.7030 J/m^2
Relaxed S   = 0.6192 J/m^2
Relaxation energy = 0.0838 J/m^2
